# Notebook 06 — XGBoost Fusion Classifier

Combines 63 stylometric features with 3 ModernBERT confidence scores into a 66-dimensional feature vector and trains an XGBoost classifier for three-class phishing detection.

## What This Notebook Does
- Loads stylometric features and ModernBERT confidence scores
- Merges into a 66-dimensional feature vector per email
- Applies stratified 70/15/15 train/validation/test split
- Applies class weights to handle mild class imbalance
- Trains XGBoost classifier with 300 estimators
- Evaluates on held-out test set with full classification report
- Saves trained model to disk
- Generates confusion matrix, metrics bar chart, feature importance,and ROC curve visualisations

## Inputs
- data/processed/stylometric_features_final.csv (from Notebook 02/04)
- data/processed/modernbert_features.csv (from Notebook 05b)

## Outputs
- models/xgboost_fusion.pkl — zero-shot fusion model
- models/xgboost_modernbert.pkl — ModernBERT fusion model
- results/confusion_matrices.png
- results/roc_curves.png
- results/feature_importance.png
- results/metrics_bar_chart.png

## Key Results
- Diverse Fusion Macro F1: 0.9879
- AUC-ROC: 0.9985
- MCC: 0.9820
- AI Phishing Recall: 1.000 (100%)

## XGBoost Parameters
- n_estimators: 300
- max_depth: 6
- learning_rate: 0.1
- subsample: 0.8
- colsample_bytree: 0.8
- random_state: 42

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

# Load stylometric features
stylo = pd.read_csv(DATA_PROCESSED / "stylometric_features_final.csv")
print(f"Stylometric features: {stylo.shape}")

# Load LLM features
llm = pd.read_csv(DATA_PROCESSED / "llm_features.csv")
print(f"LLM features: {llm.shape}")

# Load original dataset for reference
dataset = pd.read_csv(DATA_PROCESSED / "dataset_final.csv")
print(f"Dataset: {dataset.shape}")

print("\nStylometric label distribution:")
print(stylo['label'].value_counts().sort_index())
print("\nLLM label distribution:")
print(llm['true_label'].value_counts().sort_index())

In [ ]:
#merge features
# Extract LLM confidence scores (the 3 features we need)
llm_features = llm[['conf_legitimate', 'conf_human_phishing', 'conf_ai_phishing']].copy()

# Extract stylometric features (drop label column)
stylo_features = stylo.drop(columns=['label']).copy()

# Combine into one feature matrix
X = pd.concat([stylo_features, llm_features], axis=1)
y = stylo['label']

print(f"Combined feature matrix: {X.shape}")
print(f"Features: {X.shape[1]} (63 stylometric + 3 LLM confidence scores)")
print(f"Labels: {y.shape}")
print(f"\nClass distribution:")
print(y.value_counts().sort_index())
print(f"\nFeature columns:")
print(f"  Stylometric: {stylo_features.shape[1]}")
print(f"  LLM scores:  {llm_features.shape[1]}")
print(f"  Total:       {X.shape[1]}")

In [ ]:
# 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Dataset splits:")
print(f"  Training:   {X_train.shape[0]} emails ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"  Validation: {X_val.shape[0]} emails ({X_val.shape[0]/len(X)*100:.0f}%)")
print(f"  Test:       {X_test.shape[0]} emails ({X_test.shape[0]/len(X)*100:.0f}%)")

print(f"\nClass distribution in training set:")
print(y_train.value_counts().sort_index())
print(f"\nClass distribution in test set:")
print(y_test.value_counts().sort_index())

In [ ]:
#train XGBoost
# Calculate class weights to handle imbalance
class_counts = y_train.value_counts().sort_index()
total = len(y_train)
class_weights = {cls: total / (len(class_counts) * count)
                 for cls, count in class_counts.items()}
sample_weights = y_train.map(class_weights)

print("Class weights applied:")
for cls, weight in class_weights.items():
    label_name = {0: "Legitimate", 1: "Human phishing", 2: "AI phishing"}[cls]
    print(f"  Class {cls} ({label_name}): {weight:.3f}")

# Train XGBoost
print("\nTraining XGBoost classifier")
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Training complete.")

# Evaluate on validation set
val_preds = model.predict(X_val)
print("\nValidation Set Results:")
print(classification_report(y_val, val_preds,
      target_names=['Legitimate', 'Human Phishing', 'AI Phishing']))

In [ ]:
import pickle

model_path = MODELS_DIR / "xgboost_fusion.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"Model saved to: {model_path}")
print(f"File size: {model_path.stat().st_size / 1024:.1f} KB")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, matthews_corrcoef,
                             roc_curve, auc)
from sklearn.preprocessing import label_binarize
import time

RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Final evaluation on held-out test set
print("Evaluating on held-out test set")
test_preds = model.predict(X_test)
test_proba = model.predict_proba(X_test)

# Classification report
print("\nTest Set Results:")
print(classification_report(y_test, test_preds,
      target_names=['Legitimate', 'Human Phishing', 'AI Phishing']))

# Macro F1
from sklearn.metrics import f1_score
macro_f1 = f1_score(y_test, test_preds, average='macro')

# MCC
mcc = matthews_corrcoef(y_test, test_preds)

# AUC-ROC
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
auc_roc = roc_auc_score(y_test_bin, test_proba, multi_class='ovr', average='macro')

# False Positive Rate for legitimate class
cm = confusion_matrix(y_test, test_preds)
fp_legit = cm[1][0] + cm[2][0]
tn_legit = cm[1][1] + cm[1][2] + cm[2][1] + cm[2][2]
fpr = fp_legit / (fp_legit + tn_legit)

# AI phishing recall
ai_recall = cm[2][2] / cm[2].sum()

# Inference time
start = time.time()
_ = model.predict(X_test)
inference_time = (time.time() - start) / len(X_test) * 1000

print("\nSIX EVALUATION METRICS")
print(f"1. Macro F1-score:      {macro_f1:.4f}")
print(f"2. AUC-ROC:             {auc_roc:.4f}")
print(f"3. MCC:                 {mcc:.4f}")
print(f"4. False Positive Rate: {fpr:.4f}")
print(f"5. AI-Phishing Recall:  {ai_recall:.4f}")
print(f"6. Inference Time:      {inference_time:.4f} ms/email")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Legitimate', 'Human Phishing', 'AI Phishing'],
    yticklabels=['Legitimate', 'Human Phishing', 'AI Phishing'],
    ax=ax
)

ax.set_title('Confusion Matrix — XGBoost Fusion Classifier\n(Test Set)', 
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrix.png")

In [ ]:
metrics = {
    'Macro F1': macro_f1,
    'AUC-ROC': auc_roc,
    'MCC': mcc,
    'AI Phishing\nRecall': ai_recall,
}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(metrics.keys(), metrics.values(),
              color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D'],
              width=0.5, edgecolor='white', linewidth=1.5)

# Add value labels on bars
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Evaluation Metrics — XGBoost Fusion Classifier', 
             fontsize=14, fontweight='bold')
ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='0.90 threshold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "metrics_bar_chart.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: metrics_bar_chart.png")

In [ ]:
feature_names = list(X.columns)
importances = model.feature_importances_

# Get top 20 features
indices = np.argsort(importances)[::-1][:20]
top_features = [feature_names[i] for i in indices]
top_importances = importances[indices]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(range(20), top_importances[::-1],
               color='#2E86AB', edgecolor='white', linewidth=0.5)

ax.set_yticks(range(20))
ax.set_yticklabels(top_features[::-1], fontsize=10)
ax.set_xlabel('Feature Importance Score', fontsize=12)
ax.set_title('Top 20 Most Important Features\nXGBoost Fusion Classifier', 
             fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_importance.png")

XGBoost with Fine-tuned ModernBERT scores

In [ ]:
import pandas as pd
import numpy as np
import pickle
import xgboost as xgb
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"

# Reload stylometric features
stylo = pd.read_csv(DATA_PROCESSED / "stylometric_features_final.csv")
stylo_features = stylo.drop(columns=['label']).copy()

print("Ready")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import xgboost as xgb
import warnings
# Load ModernBERT confidence scores
modernbert = pd.read_csv(DATA_PROCESSED / "modernbert_features.csv")
print(f"ModernBERT features: {modernbert.shape}")
print(f"\nPrediction distribution:")
print(modernbert['llm_label'].value_counts().sort_index())

# Extract ModernBERT confidence scores
modernbert_features = modernbert[['conf_legitimate', 
                                   'conf_human_phishing', 
                                   'conf_ai_phishing']].copy()

# Combine with stylometric features
X_modern = pd.concat([stylo_features, modernbert_features], axis=1)
y_modern = stylo['label']

print(f"\nCombined feature matrix: {X_modern.shape}")
print(f"Features: {X_modern.shape[1]} (63 stylometric + 3 ModernBERT scores)")

# Same train/test split
X_train_m, X_temp_m, y_train_m, y_temp_m = train_test_split(
    X_modern, y_modern, test_size=0.30, random_state=42, stratify=y_modern
)
X_val_m, X_test_m, y_val_m, y_test_m = train_test_split(
    X_temp_m, y_temp_m, test_size=0.50, random_state=42, stratify=y_temp_m
)

# Class weights
class_counts_m = y_train_m.value_counts().sort_index()
total_m = len(y_train_m)
class_weights_m = {cls: total_m / (len(class_counts_m) * count)
                   for cls, count in class_counts_m.items()}
sample_weights_m = y_train_m.map(class_weights_m)

# Train XGBoost with ModernBERT scores
print("\nTraining XGBoost with ModernBERT scores")
model_modern = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

model_modern.fit(
    X_train_m, y_train_m,
    sample_weight=sample_weights_m,
    eval_set=[(X_val_m, y_val_m)],
    verbose=False
)

print("Training complete.")

# Evaluate on validation set
val_preds_m = model_modern.predict(X_val_m)
print("\nValidation Set Results (ModernBERT + Stylometric):")
print(classification_report(y_val_m, val_preds_m,
      target_names=['Legitimate', 'Human Phishing', 'AI Phishing']))

# Save model
model_modern_path = MODELS_DIR / "xgboost_modernbert.pkl"
with open(model_modern_path, 'wb') as f:
    pickle.dump(model_modern, f)
print(f"Model saved to: {model_modern_path}")